# 12-month QQQ put quotes: audit of the local hedge model + surface fit
**Stage 1 gate for the options overlay** (README "Options overlay"). Written offline, not yet run.

Upload to the Object Store first (from your laptop, repo root):
`lean cloud object-store set options_ledger_central.csv results/options_ledger_central.csv`

What this notebook does, all on QC's listed US equity-option quotes (they start in **January 2012**):
1. **Trade audit.** For every option trade the local model made in 2012-2026 (all central cells), find the real QQQ
   put with the same moneyness and ~12 months left, then follow *that contract* to the sale / expiry date.
   Compare the real **ask at entry** and **bid at exit** (as % of spot) with the model's price.
2. **Stress fills.** Show the held contracts around **Mar 2020** and **May 2022**: bid, ask, spread, volume,
   open interest on the sale dates.
3. **Contracts at $1M.** Whole-contract counts and the sizing error vs the model's notional.
4. **Surface fit.** Monthly sample 2012-2026 of puts at 80-100% moneyness and 1-15 months: fit the parameters
   of `research/options_model.Surface` (ATM ratio to VXN, long-run vol, mean reversion, put skew at 12 months,
   half-spread and its stress scaling) → `options_surface_fit.csv`. Copy it to `results/`; the local model loads
   it automatically, then re-run `python research/options_overlay.py`.

**Gate (decided before running):** stop the options track for v3 if the real entry asks are above the model's
*pessimistic* band, or the Mar 2020 / May 2022 exits show no usable bids.

In [ ]:
from QuantConnect.Research import QuantBook
import numpy as np, pandas as pd
from scipy.optimize import brentq, least_squares
from scipy.special import ndtr
qb = QuantBook()
qqq = qb.add_equity("QQQ", Resolution.DAILY, data_normalization_mode=DataNormalizationMode.RAW).symbol
vxn = qb.add_index("VXN", Resolution.DAILY).symbol
opt = qb.add_option("QQQ", Resolution.DAILY)
opt.set_filter(lambda u: u.include_weeklys().puts_only().strikes(-200, 200).expiration(0, 500))   # wide: held puts go deep ITM in crashes
CAN = opt.symbol
TBILL = {2012: .0009, 2013: .0006, 2014: .0003, 2015: .0005, 2016: .0032, 2017: .0093, 2018: .0194, 2019: .0206,
         2020: .0035, 2021: .0005, 2022: .0202, 2023: .0507, 2024: .0497, 2025: .0410, 2026: .0375}
spot = qb.history(qqq, datetime(2011, 12, 1), datetime.now(), Resolution.DAILY).loc[qqq]["close"]
spot.index = pd.to_datetime(spot.index).normalize()
vx = qb.history(vxn, datetime(2011, 12, 1), datetime.now(), Resolution.DAILY).loc[vxn]["close"] / 100
vx.index = pd.to_datetime(vx.index).normalize()
print("QQQ raw", spot.index.min().date(), "->", spot.index.max().date(), "| VXN", vx.index.min().date(), "->", vx.index.max().date())

def bs_put(S, K, T, r, s):
    v = s * np.sqrt(T); d1 = (np.log(S / K) + (r + .5 * s * s) * T) / v
    return K * np.exp(-r * T) * ndtr(-(d1 - v)) - S * ndtr(-d1)

def iv_put(px, S, K, T, r):
    try:
        return brentq(lambda s: bs_put(S, K, T, r, s) - px, 1e-3, 5.0)
    except Exception:
        return np.nan

def chain_on(day, days=4):
    """Daily put quotes on `day` (first trading day on/after it): one row per contract."""
    h = qb.option_history(CAN, day, day + timedelta(days=days), Resolution.DAILY)
    df = h.data_frame
    if df is None or df.empty:
        return pd.DataFrame()
    df = df.reset_index()
    df["time"] = pd.to_datetime(df["time"]).dt.normalize()
    first = df["time"].min()
    df = df[df["time"] == first].copy()
    df["expiry"] = pd.to_datetime(df["expiry"])
    df["dte"] = (df["expiry"] - first).dt.days
    df["mid"] = (df["bidclose"] + df["askclose"]) / 2
    return df

def pick(df, S, moneyness, dte):
    d = df[(df["askclose"] > 0)].copy()
    if d.empty:
        return None
    d["score"] = (d["strike"] / S - moneyness).abs() * 4 + (d["dte"] - dte).abs() / 365
    return d.sort_values("score").iloc[0]

## 1-3. Trade audit: follow each model trade's real contract

In [ ]:
L = pd.read_csv(qb.object_store.get_file_path("options_ledger_central.csv"), parse_dates=["date"])
L = L[L["date"] >= "2012-01-03"].reset_index(drop=True)
OPENS = {"entry", "roll buy", "roll-down buy"}
rows, held = [], {}
for _, t in L.iterrows():
    day = t["date"].to_pydatetime()
    S = float(spot.asof(t["date"]))
    df = chain_on(day)
    if df.empty:
        rows.append(dict(cell=t.cell, date=t.date.date(), event=t.event, note="no chain")); continue
    if t["event"] in OPENS:
        c = pick(df, S, t["moneyness"], t["T"] * 365.25)
        held[t.cell] = c["symbol"] if c is not None else None
        q = c
    else:
        sym = held.get(t.cell)
        q = df[df["symbol"] == sym].iloc[0] if sym is not None and (df["symbol"] == sym).any() else None
    if q is None:
        rows.append(dict(cell=t.cell, date=t.date.date(), event=t.event, note="contract not quoted")); continue
    real = q["askclose"] if t["event"] in OPENS else q["bidclose"]
    model_fill = abs(t["cash"]) / abs(t["units"]) / t["S"] if t["units"] else np.nan    # model fill as % of spot
    n_1m = round(t["notional_pct_nav"] * 1_000_000 / (100 * S)) if t["event"] in OPENS else np.nan
    rows.append(dict(cell=t.cell, date=t.date.date(), event=t.event, vxn=t.vxn, strike=q["strike"], spot=S,
                     moneyness_real=q["strike"] / S, moneyness_model=t["moneyness"], dte=q["dte"],
                     real_fill_pct=real / S, model_fill_pct=model_fill, real_mid_pct=q["mid"] / S,
                     model_mid_pct=t["mid_pct_spot"], spread_pct_mid=(q["askclose"] - q["bidclose"]) / q["mid"] if q["mid"] > 0 else np.nan,
                     volume=q.get("volume", np.nan), open_interest=q.get("openinterest", np.nan), contracts_at_1M=n_1m,
                     sizing_error=(n_1m * 100 * S / 1_000_000 - t["notional_pct_nav"]) if t["event"] in OPENS else np.nan))
A = pd.DataFrame(rows)
A["real_over_model"] = A["real_fill_pct"] / A["model_fill_pct"]
qb.object_store.save("results/options_quotes_12m.csv", A.to_csv(index=False))
print(A.groupby("event")[["real_over_model", "spread_pct_mid"]].describe().round(3).T)
A.head(20)

**Gate check.** Entry asks: compare `real_over_model` for opening trades with the model's pessimistic band
(the local pessimistic case pays about 1.26× the central price for a 12-month 85% put at VXN 14-25; recompute it
with `research/options_model.py` if the surface changes).

In [ ]:
opens = A[A["event"].isin(OPENS)]
print("entry ask / model fill: median %.2f, 90th pct %.2f, n=%d" % (opens.real_over_model.median(),
      opens.real_over_model.quantile(.9), len(opens)))
for a, b in [("2020-02-20", "2020-04-15"), ("2022-04-20", "2022-06-30")]:
    print(f"\n--- {a} -> {b}")
    print(A[(pd.to_datetime(A.date) >= a) & (pd.to_datetime(A.date) <= b)]
          [["cell", "date", "event", "strike", "dte", "real_fill_pct", "model_fill_pct", "spread_pct_mid", "volume", "open_interest"]]
          .round(4).to_string())

## 4. Surface fit (monthly sample 2012-2026)

In [ ]:
grid_m = [0.80, 0.85, 0.90, 0.95, 1.00]
grid_d = [30, 91, 182, 273, 365, 450]
obs = []
for day in pd.date_range("2012-01-01", datetime.now(), freq="MS"):
    df = chain_on(day.to_pydatetime(), days=6)
    if df.empty:
        continue
    t0 = df["time"].iloc[0]
    S, v, r = float(spot.asof(t0)), float(vx.asof(t0)), TBILL.get(t0.year, .02)
    for m in grid_m:
        for d in grid_d:
            c = pick(df[(df["dte"] - d).abs() <= max(20, d * .2)], S, m, d)
            if c is None or c["bidclose"] <= 0 or abs(c["strike"] / S - m) > 0.02:
                continue
            T = c["dte"] / 365
            obs.append(dict(date=t0, S=S, vxn=v, m_target=m, d_target=d, K=c["strike"], T=T,
                            iv=iv_put(c["mid"], S, c["strike"], T, r), h=(c["askclose"] - c["bidclose"]) / (2 * c["mid"])))
O = pd.DataFrame(obs).dropna(subset=["iv"])
print(len(O), "quotes;", O.date.min().date(), "->", O.date.max().date())
O.groupby(["m_target", "d_target"]).iv.median().unstack().round(3)

In [ ]:
T_REF = 30 / 365
atm = O[O.m_target == 1.00]
def atm_model(p, T, v):
    ratio, lr, kappa = p
    x = kappa * T
    return lr + (ratio * v - lr) * (1 - np.exp(-x)) / x
fit = least_squares(lambda p: atm_model(p, atm["T"].values, atm["vxn"].values) - atm["iv"].values,
                    x0=[0.92, 0.23, 2.0], bounds=([0.5, 0.05, 0.05], [1.5, 0.6, 20]))
ratio, lr, kappa = fit.x
print("ATM fit: ratio %.3f  lr %.3f  kappa %.2f  rmse %.4f" % (ratio, lr, kappa, np.sqrt(np.mean(fit.fun ** 2))))
# put skew: IV(K)/ATM - 1 = skew * sqrt(T_REF/T) * ln(S/K), solved per quote against the fitted ATM of that day
otm = O[O.m_target < 1.00].copy()
otm["atm_fit"] = atm_model(fit.x, otm["T"].values, otm["vxn"].values)
otm["skew"] = (otm["iv"] / otm["atm_fit"] - 1) / (np.sqrt(T_REF / otm["T"]) * np.log(otm["S"] / otm["K"]))
print(otm.groupby(["m_target", "d_target"]).skew.median().unstack().round(2))
skew_12m = otm[(otm.m_target == 0.85) & (otm.d_target == 365)].skew.median()
# spreads: h = half_spread x max(1, VXN / stress_vxn) for 12-month ~85% puts
p12 = O[(O.m_target == 0.85) & (O.d_target == 365)]
h0 = p12[p12.vxn < 0.25].h.median()
hi = p12[p12.vxn >= 0.25]
stress = np.median(h0 * hi.vxn * 100 / hi.h) if len(hi) else 25.0
F = pd.DataFrame([("ratio", ratio), ("lr", lr), ("kappa", kappa), ("skew_put", skew_12m), ("half_spread", h0),
                  ("stress_vxn", stress)], columns=["param", "value"])
F["note"] = "fit on QC QQQ put quotes 2012-2026 (skew_put at 12 months, 85% strike)"
qb.object_store.save("results/options_surface_fit.csv", F.to_csv(index=False))
qb.object_store.save("results/options_surface_obs.csv", O.to_csv(index=False))
F

## 5. Download and re-run locally
Download `results/options_quotes_12m.csv`, `results/options_surface_fit.csv` (and `options_surface_obs.csv`) from the
Object Store into the repo's `results/`, then `python research/options_overlay.py && python research/options_figs.py`.
The trials table then says `surface: fit: options_surface_fit.csv`.